# DRLB: двухэпоховый тренинг (two-epoch fit)

**Гипотеза:** два прохода по обучающему датасету улучшают качество модели.

**Техника:** вызываем `bidder.fit(train_stats_df, train_campaigns_df)` дважды подряд на одном и том же объекте `DRLBBidder` — без изменений существующего кода.

**Что сохраняется между эпохами:**
- Веса DQN и RewardNet (обучение продолжается с той точки, где остановилось)
- `agent.global_T` → epsilon уже упирается в `eps_end ≈ 0.05` на старте эпохи 2 (жадная политика)
- Оптимизаторы Adam (состояние моментов)

**Что сбрасывается:** lambda в начале каждого эпизода возвращается к `train_prior_lambda_init` (то же, что и в эпохе 1).

Никаких изменений в `DRLBBidder`, `RlBidAgent`, `simulate` или адаптерах.

In [1]:
import sys
import pickle
from dataclasses import replace
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = "/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from example_notebooks.experiments.adapters.drlb_adapter import (
    DRLB_RUNTIME_DEFAULTS,
    add_smoothed_diagnostics,
)
from example_notebooks.experiments.adapters.drlb_adapter import summarize_diagnostics
from example_notebooks.experiments.drlb.profiles import build_config as build_drlb_config
from example_notebooks.experiments.drlb.profiles import get_profile as get_drlb_profile
from example_notebooks.experiments.infra.artifacts import score_to_dict
from example_notebooks.experiments.infra.split_utils import resolve_normalized_splits
from simulator.model.drlb_bidder import DRLBBidder
from simulator.model.linear_bidder import LinearBidder
from simulator.simulation.simulate import simulate_campaign
from simulator.validation.check_results import autobidder_check, create_campaign_instance

/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RUN_NAME = "drlb_two_epoch_fit"
DRLB_PROFILE = "drlb_smooth"
SPLIT_SET = "full_train_val_holdout"
N_EPOCHS = 2

OUTPUTS_DIR = Path(REPO_ROOT) / "example_notebooks" / "experiments" / "drlb" / RUN_NAME / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
_linear_pkl = (
    Path(REPO_ROOT)
    / "example_notebooks"
    / "evaluate_baselines"
    / "best_params"
    / "fpa_baseline_n10_rndm_42"
    / "linear_scr_FPA.pkl"
)
with _linear_pkl.open("rb") as f:
    linear_tuned_params = pickle.load(f)
linear_tuned_params

{'coef': 0.023335213830958296,
 'lower_clip': 9,
 'upper_clip': 1,
 'factor': 3.4224852046754637}

In [4]:
config = build_drlb_config(RUN_NAME, profile=DRLB_PROFILE, split_set=SPLIT_SET)
config = replace(config, max_steps=None)  # train on all timesteps

normalized_splits = resolve_normalized_splits(config)
normalized_splits

{'train': {'campaigns_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/campaigns_fpa_train_val.csv',
  'stats_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/stats_fpa_train_val.csv'},
 'val': {'campaigns_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/campaigns_fpa_val_val.csv',
  'stats_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/stats_fpa_val_val.csv'},
 'test_holdout': {'campaigns_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/campaigns_fpa_holdout_test.csv',
  'stats_path': '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/data/fpa/stats_fpa_holdout_test.csv'}}

In [5]:
train_stats_df = pd.read_csv(normalized_splits["train"]["stats_path"])
train_campaigns_df = pd.read_csv(normalized_splits["train"]["campaigns_path"])

print(f"Train campaigns : {len(train_campaigns_df):,}")
print(f"Train stats rows: {len(train_stats_df):,}")

MEAN_CLICK_PRICE = 5.0

{'model_path': None,
 'eval_mode': True,
 'debug_logs': False,
 'fit_log_every': 500,
 'inference_log_every': 24,
 'max_bid': 100.0,
 'T': 72,
 'lambda_min': 1e-06,
 'lambda_max': 10.0,
 'bids_per_timestep': 1,
 'dqn_soft_update_tau': 0.01,
 'dqn_loss_type': 'smooth_l1',
 'dqn_grad_clip_norm': 5.0,
 'dqn_reward_clip_value': 10.0,
 'reward_net_loss_type': 'smooth_l1',
 'reward_net_grad_clip_norm': 5.0,
 'reward_net_reward_clip_value': 10.0,
 'dqn_gamma': 1.0,
 'dqn_lr': 0.0001,
 'dqn_target_update_interval': 100,
 'reward_net_lr': 0.001,
 'dqn_epsilon_start': 0.95,
 'dqn_epsilon_end': 0.05,
 'dqn_epsilon_anneal': 2e-05,
 'state_type': 'improved',
 'objective': 'clicks',
 'inference_lambda_init_mode': 'checkpoint_final',
 'verbose': False,
 'use_tqdm': True}

In [6]:
# Прогон LinearBidder на train → биды → linear_lambda = ctr_pred / bid
# (по аналогии с drlb_with_linear_lambda_fit_checkpoint_infer.ipynb)
linear_hist_parts = []
for _, campaign_row in train_campaigns_df.iterrows():
    campaign_id = int(campaign_row["campaign_id"])
    campaign_stats = train_stats_df[train_stats_df.campaign_id == campaign_id].copy()
    if campaign_stats.empty:
        continue
    campaign = create_campaign_instance(campaign_row, MEAN_CLICK_PRICE)
    bidder_lin = LinearBidder(linear_tuned_params)
    history = simulate_campaign(
        campaign=campaign,
        bidder=bidder_lin,
        stats_file=campaign_stats,
        auction_mode=config.auction_mode,
    )
    hist_df = history.to_data_frame()
    if not hist_df.empty:
        linear_hist_parts.append(hist_df)

linear_hist = pd.concat(linear_hist_parts, ignore_index=True)

ctr_by_period = (
    train_stats_df
    .groupby(["campaign_id", "period"], as_index=False)
    .agg(ctr_pred=("CTRPredicts", "mean"))
)
lambda_df = linear_hist.merge(
    ctr_by_period,
    left_on=["campaign_id", "prev_timestamp"],
    right_on=["campaign_id", "period"],
    how="inner",
)
lambda_df = lambda_df[(lambda_df["bid"] > 0) & (lambda_df["ctr_pred"] > 0)].copy()
lambda_df["linear_lambda"] = lambda_df["ctr_pred"] / lambda_df["bid"]

linear_lambda_init = float(lambda_df["linear_lambda"].mean())
print(f"linear_lambda_init (mean): {linear_lambda_init:.6f}")
print(f"linear_lambda median:      {float(lambda_df['linear_lambda'].median()):.6f}")

Train campaigns : 1,027
Train stats rows: 1,246,481


In [ ]:
profile_data = get_drlb_profile(DRLB_PROFILE)

bidder_params = {
    **DRLB_RUNTIME_DEFAULTS,
    **profile_data["base_drlb_params"],
    **profile_data["reference_model_params"],
    "state_type": profile_data["state_type"],
    "objective": profile_data["objective"],
    "fit_lambda_init": linear_lambda_init,
    "inference_lambda_init": None,
    "inference_lambda_init_mode": "checkpoint_final",
    "verbose": False,
    "use_tqdm": True,
}
bidder_params

In [7]:
bidder = DRLBBidder(bidder_params)

# Запоминаем границу после каждой эпохи по количеству шагов в step_memory
epoch_boundaries: list[int] = []

for epoch in range(1, N_EPOCHS + 1):
    print(f"\n{'='*55}")
    print(f"  Epoch {epoch}/{N_EPOCHS}")
    print(f"{'='*55}")
    bidder.fit(train_stats_df, campaigns_df=train_campaigns_df)
    epoch_boundaries.append(len(bidder.agent.step_memory))
    print(f"  global_T after epoch {epoch}: {bidder.agent.global_T:,}")
    print(f"  eps after epoch {epoch}:      {bidder.agent.eps:.4f}")
    print(f"  lambda after epoch {epoch}:   {bidder.agent.ctl_lambda:.6f}")

print(f"\nEpoch boundaries (step_memory idx): {epoch_boundaries}")


  Epoch 1/2


DRLBBidder.fit:  96%|█████████▌| 46338/48240 [02:42<00:06, 285.08step/s]


  global_T after epoch 1: 46,338
  eps after epoch 1:      0.0500
  lambda after epoch 1:   0.000007

  Epoch 2/2


DRLBBidder.fit:  96%|█████████▌| 46315/48240 [02:43<00:06, 283.98step/s]


  global_T after epoch 2: 92,653
  eps after epoch 2:      0.0500
  lambda after epoch 2:   0.000018

Epoch boundaries (step_memory idx): [46338, 92653]


In [8]:
diagnostics_df = bidder.get_training_diagnostics()
print(f"Total training steps: {len(diagnostics_df):,}")
print(f"Epoch boundaries:     {epoch_boundaries}")
diagnostics_df.tail()

Total training steps: 92,653
Epoch boundaries:     [46338, 92653]


,global_t,rem_budget,lambda,eps,dqn_action,dqn_loss,reward_signal,reward_net_loss
92648,92649,603,0.000015,0.05,6,19.818213,0.635679,1.549462
92649,92650,601,0.000016,0.05,6,34.463245,0.561858,0.834700
92650,92651,601,0.000016,0.05,1,14.292465,0.728002,1.136847
92651,92652,601,0.000017,0.05,6,7.693017,0.415757,1.682916
92652,92653,600,0.000018,0.05,6,18.549158,0.364612,0.940964


In [9]:
plot_df = add_smoothed_diagnostics(diagnostics_df, smoothing_window=500)
x = plot_df["global_t"].to_numpy() if "global_t" in plot_df.columns else plot_df.index.to_numpy()

# Вертикальные линии на границах эпох (все, кроме последней)
epoch_vlines = [plot_df["global_t"].iloc[b - 1] for b in epoch_boundaries[:-1] if b > 0 and b <= len(plot_df)]

fig, axes = plt.subplots(4, 1, figsize=(14, 16), dpi=130, sharex=True)

# --- DQN Loss ---
ax = axes[0]
ax.plot(x, plot_df["dqn_loss"], linewidth=0.8, alpha=0.2, color="#1f77b4")
if "dqn_loss_smooth" in plot_df.columns:
    ax.plot(x, plot_df["dqn_loss_smooth"], linewidth=1.8, color="#1f77b4", label="DQN loss (smooth)")
for vx in epoch_vlines:
    ax.axvline(vx, color="red", linestyle="--", linewidth=1.2, alpha=0.7, label="epoch boundary")
ax.set_title("DQN Loss (Train)")
ax.set_ylabel("loss")
ax.legend(loc="upper right")

# --- RewardNet Loss ---
ax = axes[1]
ax.plot(x, plot_df["reward_net_loss"], linewidth=0.8, alpha=0.2, color="#2ca02c")
if "reward_net_loss_smooth" in plot_df.columns:
    ax.plot(x, plot_df["reward_net_loss_smooth"], linewidth=1.8, color="#2ca02c", label="RewardNet loss (smooth)")
for vx in epoch_vlines:
    ax.axvline(vx, color="red", linestyle="--", linewidth=1.2, alpha=0.7)
ax.set_title("RewardNet Loss (Train)")
ax.set_ylabel("loss")
ax.legend(loc="upper right")

# --- Reward Signal ---
ax = axes[2]
ax.plot(x, plot_df["reward_signal"], linewidth=0.8, alpha=0.2, color="#ff7f0e")
if "reward_signal_smooth" in plot_df.columns:
    ax.plot(x, plot_df["reward_signal_smooth"], linewidth=1.8, color="#ff7f0e", label="reward signal (smooth)")
for vx in epoch_vlines:
    ax.axvline(vx, color="red", linestyle="--", linewidth=1.2, alpha=0.7)
ax.set_title("Reward Signal (Train)")
ax.set_ylabel("reward")
ax.legend(loc="upper right")

# --- Epsilon ---
ax = axes[3]
ax.plot(x, plot_df["eps"], linewidth=1.2, color="#9467bd", label="epsilon")
for vx in epoch_vlines:
    ax.axvline(vx, color="red", linestyle="--", linewidth=1.2, alpha=0.7)
ax.set_title("Epsilon (ε)")
ax.set_xlabel("global_t")
ax.set_ylabel("epsilon")
ax.legend(loc="upper right")

fig.suptitle(f"DRLB Two-Epoch Training Diagnostics (N_EPOCHS={N_EPOCHS})\nRed dashed = epoch boundary")
fig.tight_layout()

plot_path = OUTPUTS_DIR / "two_epoch_diagnostics.png"
fig.savefig(plot_path, bbox_inches="tight")
plt.show()
print(f"Plot saved: {plot_path}")

Plot saved: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/drlb/drlb_two_epoch_fit/outputs/two_epoch_diagnostics.png


/var/folders/ht/mcd64cts6p959c6g8xqp_jh00000gn/T/ipykernel_14781/1525377561.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# Сохраняем чекпоинт двухэпохового bidder
model_path = OUTPUTS_DIR / "two_epoch_model.pt"
bidder.save_model(str(model_path))
print(f"Model saved: {model_path}")

# Инференс на val
eval_params_val = {
    **bidder_params,
    "use_tqdm": False,
    "model_path": str(model_path),
    "eval_mode": True,
    "input_campaigns": normalized_splits["val"]["campaigns_path"],
    "input_stats": normalized_splits["val"]["stats_path"],
}
result_val_2ep = autobidder_check(
    bidder=DRLBBidder,
    params=eval_params_val,
    auction_mode=config.auction_mode,
    verbose=False,
    log_every_campaigns=100,
)
metrics_val_2ep = score_to_dict(
    result_val_2ep["score"],
    skipped_campaigns=result_val_2ep.get("skipped_campaigns"),
    time_inference_sec=result_val_2ep.get("time_inference_sec"),
    time_overall_sec=result_val_2ep.get("time_overall_sec"),
    average_end_balance_share=result_val_2ep.get("average_end_balance_share"),
)
metrics_val_2ep.update(summarize_diagnostics(diagnostics_df))
metrics_val_2ep["label"] = "two_epoch_val"
print("Val metrics (2 epochs):")
for k, v in metrics_val_2ep.items():
    print(f"  {k}: {v}")

Model saved: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/drlb/drlb_two_epoch_fit/outputs/two_epoch_model.pt
Val metrics (2 epochs):
  cpc_relative: 82.50894111424357
  rmse: 8.50883477815944
  clicks_sum: 517.9672878166717
  quickspend: 0.27626459143968873
  skipped_campaigns: 0
  time_inference_sec: 11.279226779937744
  time_overall_sec: 15.846557855606079
  average_end_balance_share: 0.07115756780943934
  train_steps: 92653
  last_dqn_loss: 18.549158096313477
  last_reward_net_loss: 0.9409640431404114
  dqn_loss_mean: 19.048702382028345
  dqn_loss_p95: 37.926414489746094
  reward_net_loss_mean: 1.1474305803555596
  reward_net_loss_p95: 1.7014814972877499
  reward_signal_mean: 5.092519108580521
  lambda_final: 1.812439122260682e-05
  label: two_epoch_val


In [11]:
# Baseline: ровно 1 эпоха с теми же bidder_params (для честного сравнения)
import tempfile

bidder_1ep = DRLBBidder(bidder_params)
bidder_1ep.fit(train_stats_df, campaigns_df=train_campaigns_df)
diag_1ep = bidder_1ep.get_training_diagnostics()
print(f"1-epoch training steps: {len(diag_1ep):,}")

with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as tmp:
    tmp_model_path = tmp.name
bidder_1ep.save_model(tmp_model_path)

eval_params_1ep = {
    **bidder_params,
    "use_tqdm": False,
    "model_path": tmp_model_path,
    "eval_mode": True,
    "input_campaigns": normalized_splits["val"]["campaigns_path"],
    "input_stats": normalized_splits["val"]["stats_path"],
}
result_val_1ep = autobidder_check(
    bidder=DRLBBidder,
    params=eval_params_1ep,
    auction_mode=config.auction_mode,
    verbose=False,
    log_every_campaigns=100,
)
metrics_val_1ep = score_to_dict(
    result_val_1ep["score"],
    skipped_campaigns=result_val_1ep.get("skipped_campaigns"),
    time_inference_sec=result_val_1ep.get("time_inference_sec"),
    time_overall_sec=result_val_1ep.get("time_overall_sec"),
    average_end_balance_share=result_val_1ep.get("average_end_balance_share"),
)
metrics_val_1ep.update(summarize_diagnostics(diag_1ep))
metrics_val_1ep["label"] = "one_epoch_val"
print("Val metrics (1 epoch):")
for k, v in metrics_val_1ep.items():
    print(f"  {k}: {v}")

DRLBBidder.fit:  11%|█         | 5159/48240 [00:17<02:25, 296.52step/s]DRLBBidder.fit:  16%|█▋        | 7844/48240 [00:27<02:20, 287.08step/s]DRLBBidder.fit:  69%|██████▉   | 33208/48240 [01:52<00:52, 286.61step/s]DRLBBidder.fit:  73%|███████▎  | 35164/48240 [01:59<00:50, 257.63step/s]DRLBBidder.fit:  77%|███████▋  | 36990/48240 [02:06<00:39, 284.39step/s]DRLBBidder.fit:  82%|████████▏ | 39777/48240 [02:19<00:30, 278.12step/s]DRLBBidder.fit:  87%|████████▋ | 42167/48240 [02:29<00:24, 249.95step/s]DRLBBidder.fit:  95%|█████████▍| 45807/48240 [02:44<00:08, 279.06step/s]


KeyboardInterrupt: 

In [ ]:
# Итоговое сравнение 1 эпоха vs 2 эпохи
compare_keys = [
    "clicks_sum",
    "cpc_relative",
    "quickspend",
    "rmse",
    "average_end_balance_share",
    "train_steps",
    "dqn_loss_mean",
    "reward_net_loss_mean",
    "reward_signal_mean",
    "lambda_final",
]

rows = []
for m in [metrics_val_1ep, metrics_val_2ep]:
    row = {"label": m.get("label", "?")}
    for k in compare_keys:
        row[k] = m.get(k)
    rows.append(row)

comparison_df = pd.DataFrame(rows).set_index("label")

# Дельта (2ep - 1ep) и дельта в %
delta_row = {}
for col in comparison_df.columns:
    v1 = comparison_df.loc["one_epoch_val", col]
    v2 = comparison_df.loc["two_epoch_val", col]
    if isinstance(v1, (int, float)) and isinstance(v2, (int, float)) and v1 is not None and v2 is not None:
        pct = ((v2 - v1) / abs(v1) * 100) if v1 != 0 else float("nan")
        delta_row[col] = f"{v2 - v1:+.4f} ({pct:+.1f}%)"
    else:
        delta_row[col] = "N/A"
comparison_df.loc["delta (2ep-1ep)"] = delta_row

print("=== Val: 1 epoch vs 2 epochs ===")
comparison_df

In [ ]:
# Сохраняем метрики
import json

summary = {
    "run_name": RUN_NAME,
    "drlb_profile": DRLB_PROFILE,
    "split_set": SPLIT_SET,
    "n_epochs": N_EPOCHS,
    "epoch_boundaries": epoch_boundaries,
    "one_epoch_val": {k: metrics_val_1ep.get(k) for k in compare_keys},
    "two_epoch_val": {k: metrics_val_2ep.get(k) for k in compare_keys},
}

summary_path = OUTPUTS_DIR / "comparison_summary.json"
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=True))
print(f"Summary saved: {summary_path}")
print(json.dumps(summary, indent=2))